In [ ]:
# ========== 6. LSTM МОДЕЛЬ ==========
print("\n" + "="*60)
print("ЭТАП 2: РЕАЛИЗАЦИЯ LSTM МОДЕЛИ")
print("="*60)

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Эмбеддинги
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout)
        
        # LSTM
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Выходные слои
        self.lstm_dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # Инициализация весов
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
    
    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        emb = self.embedding_dropout(emb)
        
        lstm_out, hidden = self.lstm(emb, hidden)
        lstm_out = self.lstm_dropout(lstm_out)
        lstm_out = self.layer_norm(lstm_out)
        
        logits = self.fc(lstm_out)
        return logits, hidden
    
    def generate(self, start_tokens, max_length=20, temperature=0.8, top_k=50, top_p=0.9):
        self.eval()
        with torch.no_grad():
            current = start_tokens.unsqueeze(0).to(config.device)
            generated = current.clone()
            hidden = None
            
            for _ in range(max_length):
                logits, hidden = self.forward(current, hidden)
                logits = logits[0, -1, :] / temperature
                
                # Top-k фильтрация
                if top_k > 0:
                    top_k_values, top_k_indices = torch.topk(logits, top_k)
                    probs = torch.zeros_like(logits).fill_(-float('Inf'))
                    probs[top_k_indices] = top_k_values
                    logits = probs
                
                # Top-p фильтрация
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    logits[indices_to_remove] = -float('Inf')
                
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)
                
                generated = torch.cat([generated, next_token], dim=1)
                current = next_token
            
            return generated.squeeze(0)

# Создаем модель
model = LSTMLanguageModel(
    vocab_size=len(vocab),
    embedding_dim=config.lstm_embedding_dim,
    hidden_dim=config.lstm_hidden_dim,
    num_layers=config.lstm_num_layers,
    dropout=config.lstm_dropout
).to(config.device)

print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ========== 7. ТРЕНИРОВКА LSTM ==========
print("\n" + "="*60)
print("ЭТАП 3: ТРЕНИРОВКА LSTM МОДЕЛИ")
print("="*60)

criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=config.lstm_lr, weight_decay=config.lstm_weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2)

train_losses = []
val_losses = []

best_val_loss = float('inf')

for epoch in range(config.lstm_epochs):
    # Тренировка
    model.train()
    total_train_loss = 0
    start_time = time.time()
    
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.lstm_epochs} [Train]')
    for batch_idx, batch in enumerate(progress):
        indices = batch['indices'].to(config.device, non_blocking=True)
        
        # Ограничиваем длину для экономии памяти
        if indices.size(1) > config.max_seq_len:
            indices = indices[:, :config.max_seq_len]
        
        input_seq = indices[:, :-1]
        target_seq = indices[:, 1:]
        
        optimizer.zero_grad()
        logits, _ = model(input_seq)
        loss = criterion(logits.reshape(-1, logits.size(-1)), target_seq.reshape(-1))
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        progress.set_postfix({'loss': f'{loss.item():.3f}'})
        
        # Очистка памяти
        del logits, loss, input_seq, target_seq, indices
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
    
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Валидация
    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.lstm_epochs} [Val]'):
            indices = batch['indices'].to(config.device, non_blocking=True)
            
            if indices.size(1) > config.max_seq_len:
                indices = indices[:, :config.max_seq_len]
            
            input_seq = indices[:, :-1]
            target_seq = indices[:, 1:]
            
            logits, _ = model(input_seq)
            loss = criterion(logits.reshape(-1, logits.size(-1)), target_seq.reshape(-1))
            total_val_loss += loss.item()
            
            del logits, loss, input_seq, target_seq, indices
    
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Сохраняем лучшую модель
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_lstm_model.pt')
        print(f"  ✓ Сохранена лучшая модель (val_loss: {avg_val_loss:.4f})")
    
    epoch_time = time.time() - start_time
    print(f"\nEpoch {epoch+1} Results (time: {epoch_time:.1f}s):")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Perplexity: {np.exp(avg_val_loss):.2f}")
    
    # Очистка после эпохи
    torch.cuda.empty_cache()
    gc.collect()

# Загружаем лучшую модель
model.load_state_dict(torch.load('best_lstm_model.pt', map_location=config.device))

In [ ]:
# ========== 8. ROUGE МЕТРИКИ ДЛЯ LSTM ==========
print("\n" + "="*60)
print("ЭТАП 3 (продолжение): ОЦЕНКА ROUGE ДЛЯ LSTM")
print("="*60)

def compute_rouge_lstm(model, dataloader, num_samples=100):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
    rouge1_scores = []
    rouge2_scores = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx * config.lstm_batch_size >= num_samples:
                break
                
            indices = batch['indices'].to(config.device)
            
            for i in range(len(indices)):
                text_indices = indices[i]
                text_indices = text_indices[text_indices != 0]
                
                if len(text_indices) < 10:
                    continue
                
                # Контекст (первые 70%) и референс (30%)
                split = int(len(text_indices) * 0.7)
                context = text_indices[:split]
                reference = text_indices[split:]
                
                if len(reference) < 2:
                    continue
                
                # Генерация
                generated = model.generate(context, max_length=len(reference), 
                                         temperature=0.8, top_k=50)
                generated_part = generated[split:]
                
                # Преобразуем в текст
                ref_text = ' '.join([idx_to_word.get(int(i), '') for i in reference.cpu()])
                gen_text = ' '.join([idx_to_word.get(int(i), '') for i in generated_part.cpu()])
                
                if ref_text and gen_text:
                    scores = scorer.score(ref_text, gen_text)
                    rouge1_scores.append(scores['rouge1'].fmeasure)
                    rouge2_scores.append(scores['rouge2'].fmeasure)
    
    return {
        'rouge1': np.mean(rouge1_scores) if rouge1_scores else 0,
        'rouge2': np.mean(rouge2_scores) if rouge2_scores else 0
    }

# Оценка на тесте
print("Оценка LSTM на тестовой выборке...")
lstm_rouge = compute_rouge_lstm(model, test_loader, num_samples=200)
print(f"LSTM ROUGE-1: {lstm_rouge['rouge1']:.4f}")
print(f"LSTM ROUGE-2: {lstm_rouge['rouge2']:.4f}")